# 02 – Data Preprocessing

This notebook preprocesses the German Credit dataset and produces two cleaned datasets:
- cleaned_credit_train.csv
- cleaned_credit_test.csv

## Load the Libraries

In [ ]:
# Load the required libraries

import pandas as pd
import numpy as np

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

## Load the German Credit Datset

In [ ]:
# Load the Datset from GitHub
url = "https://raw.githubusercontent.com/KrisAimlGitHub/credit-risk-capstone/main/data/raw/german.data"
df = pd.read_csv(url, sep=' ', header=None)

print('Dataset loaded. Shape: ', df.shape)

Dataset loaded. Shape:  (1000, 21)


## Assign Column Names

he German Credit dataset contains 20 features and 1 target variable. Let's use the same Column Names given in notebook 01_eda.

In [ ]:
# Assign the Column Names
columns = [
    'checking_status',  'duration',   'credit_history',   'purpose',                 'credit_amount',
    'savings',          'employment', 'installment_rate', 'personal_status',         'other_debtors',
    'residence_since',  'property',   'age',              'other_installment_plans', 'housing',
    'existing_credits', 'job',        'dependents',       'telephone',               'foreign_worker',
    'target'
]

df.columns = columns

df.head()

,checking_status,duration,credit_history,purpose,credit_amount,savings,employment,installment_rate,personal_status,other_debtors,...,property,age,other_installment_plans,housing,existing_credits,job,dependents,telephone,foreign_worker,target
0,A11,6,A34,A43,1169,A65,A75,4,A93,A101,...,A121,67,A143,A152,2,A173,1,A192,A201,1
1,A12,48,A32,A43,5951,A61,A73,2,A92,A101,...,A121,22,A143,A152,1,A173,1,A191,A201,2
2,A14,12,A34,A46,2096,A61,A74,2,A93,A101,...,A121,49,A143,A152,1,A172,2,A191,A201,1
3,A11,42,A32,A42,7882,A61,A74,2,A93,A103,...,A122,45,A143,A153,1,A173,2,A191,A201,1
4,A11,24,A33,A40,4870,A61,A73,3,A93,A101,...,A124,53,A143,A153,2,A173,2,A191,A201,2


## Save 'age' column separately.

During the later part of the project where fairness analysis is to be performed on Age Group, there arises a problem in correctly identifying the Age due to Scalling on this feature. So, let's save the 'age' separately and split this into age_train and age_test during Train/Test Split of the scaled data. This will enable us to pick the Age for Test datapoints correclt during the Fairness Analysis on Age.

In [ ]:
# Extract Age into df_age

df_age = df[['age']]
df_age.head()

,age
0,67
1,22
2,49
3,45
4,53


## Identify Features and Target Variable

In [ ]:
# X is features
X = df.drop('target', axis=1)

# y is Target
y = df['target']

## Identify Categorical and Numeric Features

Categorical variables will be one-hot encoded.

Numeric variables will be scaled.

In [ ]:
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

print('Çategorical Columns: ', categorical_cols)
print('Numerical Columns:   ', numeric_cols)


Çategorical Columns:  ['checking_status', 'credit_history', 'purpose', 'savings', 'employment', 'personal_status', 'other_debtors', 'property', 'other_installment_plans', 'housing', 'job', 'telephone', 'foreign_worker']
Numerical Columns:    ['duration', 'credit_amount', 'installment_rate', 'residence_since', 'age', 'existing_credits', 'dependents']


## Preprocessing Pipeline

- OneHotEncoder for categorical variables  
- StandardScaler for numeric variables  
- ColumnTransformer to combine both  


In [ ]:
# OneHotEncoder for Categorical Variables
categorical_transformer = OneHotEncoder(handle_unknown='ignore')

# Standard Scaler for NUmeric Variables
numeric_transformer = StandardScaler()

# Combine both the Transformed variables
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer, categorical_cols),
        ('num', numeric_transformer, numeric_cols)
    ]
)


## Train-Test Split

Let's use 70/30 split as defined in the project plan.

Note: Include the df_age as well to create age_train/test set as well.

In [ ]:
X_train, X_test, y_train, y_test, age_train, age_test = train_test_split(
    X, y, df_age, test_size=0.30, random_state=42, stratify=y
)

print('X Train Shape:   ', X_train.shape)
print('X Test Shape:    ', X_test.shape)
print('\n')
print('Age Train Shape: ', age_train.shape)
print('Age Test Shape:  ', age_test.shape)

X Train Shape:    (700, 20)
X Test Shape:     (300, 20)


Age Train Shape:  (700, 1)
Age Test Shape:   (300, 1)


## Fit the Preprocessor on Training and Test Data

Note: We are transforming the train and test data separately so that we can avoid data leakage from test set into train set (which will happen if we tranform th whole dataset first and then split it into train/test sets).

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

print('Shape of Processed X Train: ', X_train_processed.shape)
print('Shape of Processed X Test:  ', X_test_processed.shape)


Shape of Processed X Train:  (700, 61)
Shape of Processed X Test:   (300, 61)


## Convert Processed Arrays to DataFrames

This will help us convert the Cleaned / Preprocessed Data into DataFrames which can be further saved into individual CSV's for next task.

In [ ]:
# Get encoded column names
encoded_cat_cols = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_cols)
processed_cols = list(encoded_cat_cols) + numeric_cols

# DF for X Train
X_train_df = pd.DataFrame(X_train_processed, columns=processed_cols)

# DF for X Test
X_test_df  = pd.DataFrame(X_test_processed,  columns=processed_cols)

# Add target back
train_df = pd.concat([X_train_df, y_train.reset_index(drop=True)], axis=1)
test_df  = pd.concat([X_test_df,  y_test.reset_index(drop=True)],  axis=1)

# X Train Head
print('Train DF Head')
train_df.head()


Train DF Head


,checking_status_A11,checking_status_A12,checking_status_A13,checking_status_A14,credit_history_A30,credit_history_A31,credit_history_A32,credit_history_A33,credit_history_A34,purpose_A40,...,foreign_worker_A201,foreign_worker_A202,duration,credit_amount,installment_rate,residence_since,age,existing_credits,dependents,target
0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,...,1.0,0.0,-0.733512,-0.713001,0.054714,-1.660121,-0.956361,-0.724565,-0.434114,2
1,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,...,1.0,0.0,-0.231598,-0.610869,0.054714,1.076342,-1.047717,-0.724565,-0.434114,1
2,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,-0.231598,0.360687,-0.835976,-0.747967,0.048549,1.074000,2.303542,2
3,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,1.0,0.0,0.270317,-0.461601,0.945404,1.076342,-1.413139,-0.724565,-0.434114,1
4,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,-0.817165,1.506577,-1.726666,1.076342,0.322615,1.074000,-0.434114,1


## Save Cleaned Train and Test Datasets

Let's save the processed training and testing sets for use in model training.

We can save them to Colab and then manually upload them to GitHub.

Note: We can try Git Commands to save the files to GitHub directly. However this is not working die to connectivity/authorization set up. So, let's do it manually.

In [ ]:
# Save the files to Colab Env

train_df.to_csv('cleaned_credit_train.csv', index=False)
test_df.to_csv('cleaned_credit_test.csv', index=False)

In [ ]:
# Download the files.
# Note: Once downloaded, manually upload the files onto GitHub

from google.colab import files

files.download('cleaned_credit_train.csv')
files.download('cleaned_credit_test.csv')

print('Files downloaded')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Files downloaded


## Save the Age Train and Test Data to GitHub

In [ ]:
# Save the files to Colab Env

age_train.to_csv('raw_age_train.csv', index=False)
age_test.to_csv('raw_age_test.csv', index=False)

In [ ]:
# Download the files.
# Note: Once downloaded, manually upload the files onto GitHub

from google.colab import files

files.download('raw_age_train.csv')
files.download('raw_age_test.csv')

print('Files downloaded')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Files downloaded


## Summary on Data Preprocessing

- Loaded the German Credit dataset and assigned column names  

- Categorical variables were encoded using one‑hot encoding to prepare them for ML models.

- Numeric variables were scaled to ensure consistent feature ranges.

- The dataset was split into training and test sets (70/30) using a fixed random state for reproducibility.

- Raw age values were saved separately to support fairness analysis later.

- The cleaned and processed datasets were exported for use in model training.

Note: Detailed discussion on the observations will be presented in the report.